# CHEA: channel-estimation MSE on TDL-A (Fig. 3)

Reproduces Fig. 3 of the CHEA paper (arXiv:2607.16462, PIMRC 2026 Workshops):
MSE vs. SNR of the channel estimate for 4x2 MU-MIMO with 2 users on TDL-A, MCS 14,
with (a) 10 PRBs and (b) 22 PRBs.

The `*_results` files in this folder are produced by `run.sh`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress TF warnings

import sys
sys.path.append("../../../")  # repository root

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from utils.eval_plots import COLORMAP, plot_error_metric_results

results_dir = "./"  # this folder

In [ ]:
# style: [linestyle, COLORMAP index, marker, marker size, linewidth, markevery]
# (config, results-file suffix, legend label, style); {prb} is 10 or 22.
baselines = [  # evaluated for MCS indices [9, 14, 19] -> entry 1 is MCS 14
    ("baselines_4x2_chea.cfg", "_{prb}_ls",    "LS",    ["-",  2, "o", 4, 1.5, 3]),
    ("baselines_4x2_chea.cfg", "_{prb}_lmmse", "LMMSE", ["--", 4, "^", 4, 1.5, 3]),
]
models = [  # trained for MCS 14 only -> entry 0
    ("interpnet{prb}.cfg",     "",       "InterpolateNet", ["-",      0, "v", 3, 1.0, 3]),
    ("channelformer{prb}.cfg", "",       "Channelformer",  ["-",      1, "v", 3, 1.0, 3]),
    ("ha02_{prb}.cfg",         "",       "HA02",           [":",      1, "v", 3, 1.0, 3]),
    ("cevit{prb}.cfg",         "",       "CEViT",          ["-",      3, "D", 3, 1.0, 3]),
    ("chea16d.cfg",            "_{prb}", "CHEA",           ["-",      8, "s", 3, 1.0, 3]),
    ("chea64.cfg",             "_{prb}", "CHEA-XL",        ["dashed", 8, "",  3, 1.0]),
]

xlim, ylim = [-5, 30], [2e-4, 8e-1]
show = [False, False, False, False, True, True]  # only the axis ticks

fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharey=True)
for ax, prb, tag in zip(axes, (10, 22), ("a) 10 PRBs", "b) 22 PRBs")):
    for series, mcs_arr_eval_idx in ((baselines, 1), (models, 0)):
        for config, suffix, label, style in series:
            plot_error_metric_results(
                config_name=config.format(prb=prb),
                metric="mse",
                num_tx_eval=2,
                mcs_arr_eval_idx=mcs_arr_eval_idx,
                xlim=xlim,
                ylim=ylim,
                labels=[label],
                line_styles=[style],
                results_dir=[results_dir, suffix.format(prb=prb)],
                axis=ax,
                show=show,
            )
    ax.set_ylabel("MSE")
    ax.text(0.02, 0.04, tag, transform=ax.transAxes, ha="left", va="bottom", fontsize=12,
            bbox=dict(facecolor="white", edgecolor="black", linewidth=1.0,
                      boxstyle="square,pad=0.25", alpha=0.8))
axes[1].set_xlabel(r"$\mathrm{SNR (dB)}$")

legend_elements = [
    Line2D([0], [0], linewidth=2, color=COLORMAP[s[1]], linestyle=s[0], marker=s[2], label=label)
    for _, _, label, s in baselines + models
]
fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.533, 1.01), ncol=4, fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.93])

with mpl.rc_context({"pdf.fonttype": 42, "ps.fonttype": 42}):
    fig.savefig("img_tdla_mse_10_22prb_wide.png", dpi=600, bbox_inches="tight")
    fig.savefig("img_tdla_mse_10_22prb_wide.pdf", dpi=600, bbox_inches="tight")
plt.show()